# Load Dataset:

In [ ]:
import pandas as pd
df=pd.read_csv("100_Unique_QA_Dataset.csv")

In [ ]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  90 non-null     object
 1   answer    90 non-null     object
dtypes: object(2)
memory usage: 1.5+ KB


In [ ]:
# Tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace("'","")
  return text.split()


In [ ]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
# vocab
vocab = {'<UNK>': 0}

In [ ]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_token = tokenized_question + tokenized_answer
  print(merged_token)

  for token in merged_token:
    if token not in vocab:
      vocab[token]=len(vocab)

In [ ]:
df.apply(build_vocab, axis=1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
len(vocab)

324

In [ ]:
# covert words to numerical indexes
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [ ]:
text_to_indices("What is Pratik", vocab)

[1, 2, 0]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [ ]:
dataset = QADataset(df, vocab)

In [ ]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [ ]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
for question, answer in dataloader:
  print(question, answer)

tensor([[ 10,  96,   3, 104, 239]]) tensor([[240]])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([[205]])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([[110]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([[100]])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([[285]])
tensor([[ 10, 308,   3, 309, 310]]) tensor([[311]])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([[185]])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[ 1,  2,  3,  4,  5,

In [ ]:
import torch.nn as nn

In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question=self.embeddings(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output


In [ ]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
b = x(a)
c,d = y(b)
e = z(d.squeeze(0))
print(e.shape)

torch.Size([1, 324])


In [ ]:
learning_rate = 0.01
epochs = 20

In [ ]:
model = SimpleRNN(len(vocab))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# training loop
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()

    #forward pass
    output = model(question)

    # loss calculation
    loss = criterion(output, answer[0])

    # backward pass
    loss.backward()

    # update weights
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 540.762864
Epoch: 2, Loss: 313.458192
Epoch: 3, Loss: 154.499825
Epoch: 4, Loss: 67.254504
Epoch: 5, Loss: 35.064406
Epoch: 6, Loss: 26.338305
Epoch: 7, Loss: 33.953912
Epoch: 8, Loss: 25.903803
Epoch: 9, Loss: 14.306669
Epoch: 10, Loss: 19.741093
Epoch: 11, Loss: 14.801246
Epoch: 12, Loss: 16.567130
Epoch: 13, Loss: 14.326637
Epoch: 14, Loss: 11.248180
Epoch: 15, Loss: 10.294933
Epoch: 16, Loss: 4.924252
Epoch: 17, Loss: 9.737495
Epoch: 18, Loss: 4.933627
Epoch: 19, Loss: 14.470810
Epoch: 20, Loss: 13.180900


In [ ]:
def predict(model, question, threshold=0.5):
  # convert questions to numbers
  numerical_question=text_to_indices(question, vocab)

  # tensor
  tensor_question=torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(tensor_question)

  # conert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  max_prob, max_index = torch.max(probs, dim=1)

  if max_prob<threshold:
    print("I don't know the answer to that question")
  else:
    print(list(vocab.keys())[max_index])

In [ ]:
predict(model, "Who is Ronaldo?")

I don't know the answer to that question


In [ ]:
predict(model, "What is the capital of France")

paris
